In [ ]:
## مشروع ختامي: بناء خادم MCP مصغّر

في هذا المشروع، ستقوم ببناء خادم صغير يحاكي بروتوكول MCP (Model Context Protocol) باستخدام بروتوكول JSON-RPC 2.0. ستعمل بدون أي مكتبات خارجية، فقط باستخدام Python القياسي.

JSON-RPC هو بروتوكول بسيط للإجراءات عن بُعد (Remote Procedure Call) يستخدم JSON لتشفير البيانات. رسالة الطلب تحتوي على `method` (اسم الإجراء)، و `params` (وسائط الإجراء)، و `id` (معرّف الطلب). الرد يحتوي على `result` أو `error`.

سوف نقوم بتعريف أدوات (tools) بسيطة، ثم نكتب دوال لمعالجة الطلبات، وأخيراً نحاكي جلسة تبادل رسائل.

In [ ]:
## نظرة عامة على المشروع

الخطوات:

1. تعريف قاموس `TOOLS` يحتوي على أداتين: `add` (جمع عددين) و `city_info` (إرجاع معلومات عن مدينة).
2. كتابة دوال معالجة لأنواع الطلبات:
   - `initialize`: الرد بقائمة القدرات (capabilities).
   - `tools/list`: الرد بقائمة الأدوات وأوصافها.
   - `tools/call`: تنفيذ الأداة المطلوبة وإرجاع النتيجة.
3. كتابة دالة `route` لتوجيه الطلب إلى المعالج المناسب.
4. محاكاة جلسة تتبادل 4 رسائل وطباعة الردود.

In [ ]:
## هيكل رسالة JSON-RPC 2.0

**طلب (Request):**
```json
{
  "jsonrpc": "2.0",
  "method": "initialize",
  "params": {},
  "id": 1
}
```

**رد (Response):**
```json
{
  "jsonrpc": "2.0",
  "result": { ... },
  "id": 1
}
```

في حالة الخطأ:
```json
{
  "jsonrpc": "2.0",
  "error": {"code": -32601, "message": "Method not found"},
  "id": 1
}
```

In [ ]:
import json

# تعريف الأدوات (Tools)
# كل أداة لها اسم، ووصف، ودالة handler تستقبل params وتعيد النتيجة.
TOOLS = {
    "add": {
        "description": "Add two numbers a and b",
        "handler": lambda params: params["a"] + params["b"]
    },
    "city_info": {
        "description": "Get information about a city",
        "handler": lambda params: f"City {params['city']} is a major city."
    }
}

In [ ]:
## معالج التهيئة (initialize)

دالة `handle_initialize` تستقبل الطلب وترد بقائمة القدرات. يجب أن يكون الرد بالصيغة التالية:

- `protocolVersion` (إصدار البروتوكول): يمكنك استخدام "0.1.0".
- `capabilities`: كائن يحتوي على ما يدعمه الخادم. هنا سندعم `tools`.

In [ ]:
def handle_initialize(req):
    """
    معالج طلب initialize.
    """
    # TODO: أكمل الدالة بحيث تُرجع رداً يحتوي على jsonrpc=2.0 و id الطلب
    # والنتيجة تحتوي على protocolVersion و capabilities تدعم tools.
    pass

In [ ]:
## معالج tools/list

دالة `handle_tools_list` تستقبل الطلب وترد بقائمة الأدوات المتاحة. كل أداة يجب أن تحتوي على `name` (الاسم) و `description` (الوصف). استخدم قاموس `TOOLS` لاستخراج هذه المعلومات.

In [ ]:
def handle_tools_list(req):
    """
    معالج طلب tools/list.
    """
    # TODO: أعد قائمة الأدوات من TOOLS مع الاسم والوصف،
    # وضمنها في رد JSON-RPC 2.0.
    pass

In [ ]:
## معالج tools/call

دالة `handle_tools_call` تستقبل الطلب الذي يحتوي في `params` على:
- `name`: اسم الأداة المطلوبة.
- `arguments`: بارامترات الأداة (كائن).

يجب استدعاء دالة `handler` المناسبة من `TOOLS` وتمرير `arguments` إليها، ثم إرجاع النتيجة. في حالة عدم وجود الأداة أو حدوث خطأ في البارامترات، يجب إرجاع رد خطأ مناسب.

In [ ]:
def handle_tools_call(req):
    """
    معالج طلب tools/call.
    """
    # TODO: استخرج اسم الأداة و arguments من params.
    # ابحث عن الأداة في TOOLS، ثم نفذ handler، وأعد الرد.
    # تعامل مع الأخطاء.
    pass

In [ ]:
## دالة التوجيه (route)

دالة `route` هي المدخل الرئيسي لمعالجة أي طلب. تقوم بما يلي:
1. التحقق من وجود `jsonrpc` وقيمته "2.0".
2. التحقق من وجود `id`.
3. بحسب `method`، تستدعي المعالج المناسب.
4. إذا كانت method غير معروفة، ترجع خطأ "Method not found".

اكتب الدالة أدناه كاملة.

In [ ]:
def route(req):
    """
    توجيه الطلب إلى المعالج المناسب.
    """
    # التحقق من صحة الطلب
    if req.get("jsonrpc") != "2.0":
        return {
            "jsonrpc": "2.0",
            "error": {"code": -32600, "message": "Invalid Request"},
            "id": req.get("id")
        }
    if "id" not in req:
        return {
            "jsonrpc": "2.0",
            "error": {"code": -32600, "message": "Invalid Request: missing id"},
            "id": None
        }

    method = req.get("method")
    if method == "initialize":
        return handle_initialize(req)
    elif method == "tools/list":
        return handle_tools_list(req)
    elif method == "tools/call":
        return handle_tools_call(req)
    else:
        return {
            "jsonrpc": "2.0",
            "error": {"code": -32601, "message": f"Method not found: {method}"},
            "id": req["id"]
        }

In [ ]:
## محاكاة جلسة تبادل رسائل

الآن سنقوم بتجربة الخادم عن طريق إرسال مجموعة من الطلبات بالتسلسل:
1. `initialize` - لبدء الاتصال.
2. `tools/list` - لمعرفة الأدوات المتاحة.
3. `tools/call` لاستدعاء أداة `add` مع a=3, b=4.
4. `tools/call` لاستدعاء أداة `city_info` مع city='Cairo'.

كل رد يجب طباعته كـ JSON منسق.

In [ ]:
# TODO: قم بتعريف قائمة الطلبات messages كما هو موصوف في الخلية أعلاه.
# ثم لكل طلب، استدعِ route واطبع الرد كـ JSON منسق.

messages = [
    # أضف الطلبات هنا
]

for msg in messages:
    pass  # TODO: call route and print response

In [ ]:
## خاتمة

في هذا المشروع، قمنا ببناء خادم MCP مصغّر يحاكي البروتوكول باستخدام JSON-RPC 2.0 فقط مع Python القياسي. لقد تعلمنا كيفية:
- تعريف الأدوات ومعالجاتها.
- استقبال الطلبات بناءً على `method` وإرجاع الردود المناسبة.
- إدارة الأخطاء.

هذه الفكرة تمثل الأساس لبناء خوادم MCP حقيقية تتواصل مع نماذج الذكاء الاصطناعي.